In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('..')

In [4]:
from data.emission_factors import SCOPE1, SCOPE2_ELECTRICITY
from data.sites import LACQ, FRENCH_GAS_SECTOR_AVG

# Choose which site to analyse
site = LACQ

print(f"Analysing: {site['name']}")

Analysing: Lacq Gas Processing Site (illustrative)


In [6]:
def calculate_scope1(site, ef_scope1):
    """
    Calculate Scope 1 emissions from direct fuel combustion.
    Returns a dictionary of emissions by fuel type (tonnes CO2eq).
    """
    results = {}

    fuel_keys = {
        'natural_gas': 'natural_gas_MWh',
        'fuel_oil':    'fuel_oil_MWh',
        'coal':        'coal_MWh',
    }

    for fuel, site_key in fuel_keys.items():
        if site_key in site:
            mwh = site[site_key]
            ef = ef_scope1[fuel]        # kgCO₂eq / kWh
            tco2 = mwh * ef / 1000      # convert kg to tonnes
            results[fuel] = round(tco2, 1)

    if 'process_tCO2' in site:
        results['process_emissions'] = site['process_tCO2']

    results['TOTAL_scope1'] = round(sum(results.values()), 1)
    return results

scope1 = calculate_scope1(site, SCOPE1)
print('Scope 1 emissions (tCO2eq):')
for fuel, val in scope1.items():
    print(f'  {fuel:<25} {val:>10,.1f} tCO2eq')

Scope 1 emissions (tCO2eq):
  natural_gas                    181.6 tCO2eq
  fuel_oil                        16.2 tCO2eq
  TOTAL_scope1                   197.8 tCO2eq


In [8]:
def calculate_scope2(site, ef_electricity):
    """Calculate Scope 2 emissions — purchased electricity only."""
    grid = site['grid']
    ef = ef_electricity[grid]
    ef_ren = ef_electricity['renewable_ppa']
    mwh = site['electricity_MWh']

    return {
        'location_based': round(mwh * ef / 1000, 1),
        'market_based':   round(mwh * ef_ren / 1000, 1),
        'grid_ef_kgCO2_kWh': ef,
    }

scope2 = calculate_scope2(site, SCOPE2_ELECTRICITY)
print(f"Scope 2 location-based:  {scope2['location_based']:>10,.1f} tCO2eq")
print(f"Scope 2 market-based:    {scope2['market_based']:>10,.1f} tCO2eq")
print(f"Grid emission factor:    {scope2['grid_ef_kgCO2_kWh']:>10.3f} kgCO2eq/kWh")

Scope 2 location-based:        13.8 tCO2eq
Scope 2 market-based:           2.5 tCO2eq
Grid emission factor:         0.055 kgCO2eq/kWh


In [10]:
def make_summary_table(site, scope1, scope2):
    """Combine Scope 1 and 2 results into a pandas DataFrame."""
    rows = []

    for source, tco2 in scope1.items():
        if source != 'TOTAL_scope1':
            rows.append({
                'Scope':  'Scope 1',
                'Source': source.replace('_', ' ').title(),
                'tCO2eq': tco2,
                'Method': 'ADEME Base Empreinte V23.6',
            })

    rows.append({
        'Scope':  'Scope 2',
        'Source': 'Purchased electricity',
        'tCO2eq': scope2['location_based'],
        'Method': f"Location-based, {site['grid']} grid",
    })

    df = pd.DataFrame(rows)
    total = df['tCO2eq'].sum()
    df['% of total'] = (df['tCO2eq'] / total * 100).round(1)
    return df, total

df_summary, total_tco2 = make_summary_table(site, scope1, scope2)
print(df_summary.to_string(index=False))
print(f"\nTOTAL baseline: {total_tco2:,.0f} tCO2eq/year")

  Scope                Source  tCO2eq                       Method  % of total
Scope 1           Natural Gas   181.6   ADEME Base Empreinte V23.6        85.8
Scope 1              Fuel Oil    16.2   ADEME Base Empreinte V23.6         7.7
Scope 2 Purchased electricity    13.8 Location-based, FR_2023 grid         6.5

TOTAL baseline: 212 tCO2eq/year
